In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os

with zipfile.ZipFile('nuclear_webcam.zip', 'r') as zip_ref:
    zip_ref.extractall()

main_dir = 'starter_kit/'
train_df = pd.read_csv(f'{main_dir}train_data.csv', index_col='datapointID')
test_df = pd.read_csv(f'{main_dir}test_data.csv', index_col='datapointID')

train_df.head()

,file_name,label
datapointID,,
1,cc5aca4135cc4c7396bba1c32c9aa3f5.png,thorium
2,0bfd53ac789341ddaa3d3ea8c78833e7.png,thorium
3,580f87af75a742aeb20261b5163ba461.png,radium
4,6b1fa7669fc4417daf202b2fe1060d54.png,radium
5,e277e2554d264abe901542aa5585da03.png,radium


# Subatsk 1

In [45]:
import cv2

img = cv2.imread(f'{main_dir}xrays.png', 0)
img = np.array(img)
answer_s1 = np.count_nonzero(img)

answer_s1

2393

# Subtask 2

In [ ]:
from scipy.sparse import hstack, vstack, csr_matrix

def preprocess_image(img_path):
    img = cv2.imread(img_path, 0)
    _, thresh = cv2.threshold(img, 30, 255, cv2.THRESH_BINARY)

    #Feature eng
    extracted_features = {}

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)

    #exclude background
    particles_stats = stats[1:]

    extracted_features['num_particles'] = num_labels - 1

    areas = particles_stats[:, cv2.CC_STAT_AREA]
    extracted_features['min_area'] = np.min(areas)
    extracted_features['max_area'] = np.max(areas)
    extracted_features['mean_area'] = np.mean(areas)

    widths = particles_stats[:, cv2.CC_STAT_WIDTH]
    heights = particles_stats[:, cv2.CC_STAT_HEIGHT]

    extracted_features['mean_width'] = np.mean(widths)
    extracted_features['mean_height'] = np.mean(heights)
    extracted_features['elongation'] = np.mean(widths / (heights + 1e-5))

    features_df = pd.DataFrame([extracted_features])
    thresh = cv2.resize(thresh, (96, 54), cv2.INTER_AREA)  #(W, H)

    return hstack([csr_matrix(thresh.flatten()), csr_matrix(features_df)])

def get_X(df, is_train):
    rows = []

    if is_train:
        dir = f'{main_dir}train/'
    else:
        dir = f'{main_dir}test/'

    for file_name in df['file_name']:
        img_features = preprocess_image(f'{dir}{file_name}')
        rows.append(img_features)

    return vstack(rows)

X_train, y_train = get_X(train_df, is_train=True), train_df['label']
X_test = get_X(test_df, is_train=False)

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

models = {
    'LR':LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'LinearSVC':LinearSVC(max_iter=2000, class_weight='balanced', random_state=42),
    'rbf_SVC':SVC(class_weight='balanced', random_state=42),
    'RF':RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def evaluate(model):
    cv = cross_val_score(model, X_train, y_train, cv=split, scoring='f1_macro', n_jobs=-1)
    return cv.mean()

for name, model in models.items():
    model_pipe = make_pipeline(StandardScaler(with_mean=False), model)
    print(f'{name} | f1: {evaluate(model_pipe)}')


LR | f1: 0.9933291640608714
LinearSVC | f1: 0.9966645820304357
rbf_SVC | f1: 0.9899037775802858
RF | f1: 0.9933291640608714


Overfitting

In [49]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
final_model = make_pipeline(StandardScaler(with_mean=False), rf)

final_model.fit(X_train, y_train)
preds = final_model.predict(X_test)

In [50]:
output_df = pd.DataFrame({
    'subtaskID':[1] + [2] * len(test_df),
    'datapointID':[1] + list(test_df.index),
    'answer':[answer_s1] + list(preds)
})

output_df.head()

,subtaskID,datapointID,answer
0,1,1,2393
1,2,1,thorium
2,2,2,thorium
3,2,3,thorium
4,2,4,xrays


In [51]:
output_df.to_csv('subi.csv', index=False)

The official solution was to apply DFS on the pixels, but i am too lazy so I used this aproach.

86/100p; f1=0.81

A good thing to do was to give a larger image and keep it as a rectangle.